# Preparation

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

NEO4J_URI = "bolt://localhost:17687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "X"
LABELS = ["entity", "product", "brand", "category", "color"]
OLLAMA_MODEL = "qwen3-embedding:4b"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Vector Indexes

In [38]:
for label in LABELS:
    driver.execute_query(f"DROP INDEX {label}_index IF EXISTS")

In [ ]:
for label in LABELS:
    driver.execute_query(f"""
      CREATE VECTOR INDEX {label}_index IF NOT EXISTS
      FOR (e:{label}) ON (e.embedding)
      OPTIONS {{
        indexConfig: {{
          `vector.dimensions`: 2560,
          `vector.similarity_function`: 'cosine'
        }}
      }}
  """)

# Test

In [50]:
ollama_embedder = OllamaEmbeddings(model=OLLAMA_MODEL)

retriever = VectorRetriever(
    driver=driver,
    index_name="product_index",
    embedder=ollama_embedder,
    return_properties=["id"],
)

query = "Where can I find the TT-24 Spotted Trout fishing bait made by MirrOLure"

search_results = retriever.search(query_text=query, top_k=5)

print(f"Suchanfrage: '{query}'\n")
print("Ergebnisse:")
for i, result in enumerate(search_results.items, 1):
    print(i, result)
    print(f"[{i}] Score: {result.metadata.get('score'):.4f}")

Suchanfrage: 'Where can I find the TT-24 Spotted Trout fishing bait made by MirrOLure'

Ergebnisse:
1 content="{'id': '1142'}" metadata={'score': 0.934524655342102, 'nodeLabels': ['product', 'entity'], 'id': '4:e57bc706-7e95-4137-a227-c21a79593d6d:1234'}
[1] Score: 0.9345
2 content="{'id': '31150'}" metadata={'score': 0.9159144163131714, 'nodeLabels': ['product', 'entity'], 'id': '4:e57bc706-7e95-4137-a227-c21a79593d6d:34002'}
[2] Score: 0.9159
3 content="{'id': '62286'}" metadata={'score': 0.8754957914352417, 'nodeLabels': ['product', 'entity'], 'id': '4:e57bc706-7e95-4137-a227-c21a79593d6d:67990'}
[3] Score: 0.8755
4 content="{'id': '159253'}" metadata={'score': 0.8570387363433838, 'nodeLabels': ['product', 'entity'], 'id': '4:e57bc706-7e95-4137-a227-c21a79593d6d:173789'}
[4] Score: 0.8570
5 content="{'id': '45364'}" metadata={'score': 0.8539772033691406, 'nodeLabels': ['product', 'entity'], 'id': '4:e57bc706-7e95-4137-a227-c21a79593d6d:49504'}
[5] Score: 0.8540
